# 在没有 ROS 的 Python 环境中读取 D435

这个 Notebook 只使用 RSim 应用接口和原生 Cyclone DDS，可以与驱动侧使用不同的 Python 版本。先在具有 ROS 驱动的独立终端启动兼容层：

```bash
python -m rsim.drivers d435 --backend ros2
```

也可以在驱动环境的另一个 Notebook 中使用 `rsim.drivers.D435` 与 `Runtime`。两端使用相同的 `ROS_DOMAIN_ID`（未设置时为 0），不需要新建 DDS domain 或额外的数据桥。

本 Notebook 的 kernel 只需要 RSim、可正常导入的 Cyclone DDS、NumPy 和绘图依赖 `scipykit`；不需要 source ROS。Python 3.14 的 Cyclone DDS 版本兼容说明见 [DDS 文档](../docs/dds.md)。

In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path

import numpy as np
from rsim import D435, Runtime, Image, TransportConfig
from scipykit.mtp_initializer import show_image, subplots, disp, plt

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "rsim").is_dir()
    )
ASSETS = ROOT / "assets" / "dds_client"
ASSETS.mkdir(parents=True, exist_ok=True)
os.environ["NOTEBOOK_ASSETS_ROOT"] = os.path.relpath(
    ROOT / "assets", Path.cwd()
    )
os.environ["NOTEBOOK_NAME"] = "dds_client"
transport = TransportConfig(backend="cyclonedds")
print({
    "python": sys.version.split()[0], "dds_domain": transport.domain_id
    })

{'python': '3.14.7', 'dds_domain': 0}


## 连接共享源并取帧

公共 `D435()` 只连接已启动的数据源，不尝试在此环境中启动 ROS。配置应与驱动端一致；驱动端默认两路 `640x480x15`、history=8。返回值仍是 `Frame` 和 `Image`，数组为本机只读 mmap。

`get(after=...)` 等待真正的新图像；时间戳回查限定在本句柄保留的历史内。两路图像未做曝光同步或像素配准。

In [ ]:
CONFIG = dict(
    serial=os.environ.get("RSIM_D435_SERIAL", ""),
    depth_profile=os.environ.get("RSIM_D435_DEPTH_PROFILE", "640x480x15"),
    color_profile=os.environ.get("RSIM_D435_COLOR_PROFILE", "640x480x15"),
    history=8,
    transport=transport
    )
color = D435(stream="color", **CONFIG)
depth = D435(stream="depth", **CONFIG)
stamps = {"color": [], "depth": []}

async with Runtime(color, depth):
    rgb, z = await asyncio.gather(color.image.get(timeout=30), depth.image.get(timeout=30))
    for _ in range(30):
        (rgb, z) = await asyncio.gather(
            color.image.get(after=rgb.sequence, timeout=5),
            depth.image.get(after=z.sequence, timeout=5)
            )
        stamps["color"].append(rgb.stamp_ns)
        stamps["depth"].append(z.stamp_ns)
    assert await depth.image.get(timestamp_ns=z.stamp_ns, clock=z.clock) is z
    assert color.source.producer.worker_pid == depth.source.producer.worker_pid
    source_pid = color.source.producer.worker_pid

assert isinstance(rgb.data, Image) and isinstance(z.data, Image)
assert all(
    isinstance(f.data.pixels, np.memmap)
    and not f.data.pixels.flags.writeable for f in (rgb, z)
    )
assert all(len(set(s)) == 30 for s in stamps.values())
ros_imports = [
    name for name in sys.modules if name.split(".")[0] in
    ("rclpy", "sensor_msgs", "std_msgs",
        "ament_index_python") or name == "rsim.ros"
    ]
assert not ros_imports, ros_imports
report = {
    "python": sys.version.split()[0],
    "dds_domain": transport.domain_id,
    "backend": transport.backend,
    "source_pid": source_pid,
    "rgb_shape": list(rgb.data.pixels.shape),
    "depth_shape": list(z.data.pixels.shape),
    "unique_frames_per_stream": 30,
    "readonly_memmap": True,
    "timestamp_lookup": True,
    "ros_imports": ros_imports
    }
print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "python": "3.14.7",
  "dds_domain": 0,
  "backend": "cyclonedds",
  "source_pid": 474621,
  "rgb_shape": [
    480,
    640,
    3
  ],
  "depth_shape": [
    480,
    640
  ],
  "unique_frames_per_stream": 30,
  "readonly_memmap": true,
  "timestamp_lookup": true,
  "ros_imports": []
}


## 检查图像并保存

驱动端 RealSense ROS 的 `16UC1` 深度图单位为毫米，0 为无效深度。显示时转成米，保留原始数组。退出上方 `Runtime` 只释放本 Notebook 的使用租约，其他使用者仍可继续采集；已取得的映射仍可读。

In [ ]:
show_image(rgb.data.pixels, key="color")
raw = z.data.pixels
valid = raw > 0
meters = np.ma.masked_where(~valid, raw.astype(np.float32) * 0.001)
fig, ax = subplots(figsize=(6.66, 3.8))
upper = max(.1, float(np.percentile(meters.compressed(), 98))) if valid.any() else 1
im = ax.imshow(meters, cmap="viridis", vmin=0, vmax=upper)
ax.set_title("D435 depth via native DDS")
ax.set_axis_off()
fig.colorbar(im, ax=ax, label="Depth / m")
disp(fig, "depth")
plt.close(fig)
np.save(ASSETS / "color.npy", rgb.data.pixels)
np.save(ASSETS / "depth_mm.npy", raw)
(ASSETS / "capture.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

../assets/dds_client/color.png

../assets/dds_client/depth.png

291

需要计算时，可以将连接源的工厂组合进 `Map`、`Bundle` 或 `ProcessSensor`，子进程继续使用当前应用解释器和原生 DDS，不需要引入 ROS。跨层转发完整只读数组可复用同一 inode；新计算输出可用 `allocate()` 分配。描述信息由 DDS 传输，像素数组通过 mmap 共享，ROS 驱动接入环节仍可能复制数据。